## Load the domain, generate and verify tasks.

This notebook walks through the **benchmark creation** path on `domains/mock_retail`:

1. Load and validate the domain bundle (policy, DB, tools, task types, seed tasks)
2. Inspect a seed task’s oracle actions and communicate criteria
3. Replay tools in the Environment and compute a target DB hash
4. Sample constraints and generate synthetic task drafts via the LLM
5. Run the verification gate (policy rules → task-type write rules → replay) and write passing tasks to `data/benchmarks/mock_retail/tasks.json`
6. Optionally reload and re-verify saved tasks

In [10]:
import json
import pprint
from pathlib import Path

from aieng.syn_data.synbench.path_utils import find_repo_root, use_repo_root


# Set the root directory for this implementation
use_repo_root(Path("."))
ROOT = find_repo_root() / "implementations" / "agent_benchmark_generation"


DOMAIN_PATH = ROOT / "domains" / "mock_retail"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_retail"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

Implementation root: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation
Domain: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_retail
Output: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_retail


---
## Step 1 — Load and inspect the domain

A **domain bundle** is the simulated business. `load_domain()` reads all required files and returns a `DomainBundle` object used everywhere else.

In [11]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

print("Domain name:", domain.manifest.name)
print("\nTools:")
for t in domain.tools:
    print(f"  - {t.name} ({t.tool_type.value})")
print("\nTask types:", list(domain.task_types.keys()))
print("\nSeed task IDs:", [t.id for t in domain.seed_tasks])
print("\nOrders in DB:", list(domain.db["orders"].keys()))

Domain name: mock_retail

Tools:
  - find_user_id (read)
  - get_order (read)
  - list_orders (read)
  - cancel_order (write)
  - update_shipping (write)

Task types: ['inquiry', 'cancel', 'update_address', 'refuse_cancel']

Seed task IDs: ['seed_inquiry', 'seed_cancel', 'seed_refuse_cancel', 'seed_update_address']

Orders in DB: ['ord_1001', 'ord_1002', 'ord_2001']


In [12]:
# Peek at policy and one order
print("=== Policy (first 400 chars) ===")
print(domain.policy[:400], "...\n")

print("=== Sample order ord_1001 ===")
print(json.dumps(domain.db["orders"]["ord_1001"], indent=2))

=== Policy (first 400 chars) ===
# Mock Retail Customer Service Policy

You are a customer service agent for Mock Retail.

## Rules

1. Customers might not know their `user_id`. Always ask for the customer's full
   name first, then call `find_user_id` with that name to retrieve their
   `user_id`. Never invent, guess, or fabricate a `user_id`.
2. If `find_user_id` fails (user not found), tell the customer no account
   matches t ...

=== Sample order ord_1001 ===
{
  "order_id": "ord_1001",
  "user_id": "user_alice",
  "status": "pending",
  "items": [
    "widget-a"
  ],
  "shipping_address": "123 Main St, Boston, MA"
}


---
## Step 2 — Understand a Task and its oracle

Tasks are defined in `tasks.seed.json`. The **`evaluation_criteria.actions`** field is the **oracle** — the reference tool trace used for verification and scoring.

`user_scenario` keeps identity and style separate:
- **`user_name`** — customer identity (e.g. Alice Chen)
- **`personality_style`** — interaction style key from `user_simulator.yaml` (e.g. rushed)
- **`instructions`** — full clean goal with ids/args (simulator / oracle grounding; not shown to the agent)
- **`initial_message`** — styled first utterance; prefer incomplete (withhold ids/args for multi-turn)


In [13]:
cancel_task = next(t for t in domain.seed_tasks if t.id == "seed_cancel")

print("Task ID:", cancel_task.id)
print("Description:", cancel_task.description)
print("Task type:", cancel_task.task_type)
# Name and style are separate fields on the task
print("User name:", cancel_task.user_scenario.user_name)
print("Personality style:", cancel_task.user_scenario.personality_style)
print("Instructions (clean goal):", cancel_task.user_scenario.instructions)
print("Initial message (styled):", cancel_task.user_scenario.initial_message)
print("\nOracle actions:")
for i, a in enumerate(cancel_task.evaluation_criteria.actions, 1):
    print(f"  {i}. {a.name}({a.arguments})")
print("\nMust communicate:", cancel_task.evaluation_criteria.communicate_info)
print("Reward basis:", [r.value for r in cancel_task.evaluation_criteria.reward_basis])

Task ID: seed_cancel
Description: Cancel a pending order
Task type: cancel
User name: Alice Chen
Personality style: anxious
Instructions (clean goal): Request cancellation of pending order ord_1001.
Initial message (styled): I'm worried my order will ship before I can stop it — can you help?

Oracle actions:
  1. get_order({'order_id': 'ord_1001'})
  2. cancel_order({'order_id': 'ord_1001'})

Must communicate: ['canceled']
Reward basis: ['DB', 'COMMUNICATE']


---
## Step 3 — Environment: replay tools and hash the database

The **Environment** clones `db.json` and dispatches tool calls via the domain's `ToolKit`. **Replay** runs the oracle on a fresh DB; **db_hash** fingerprints the final state for scoring.

In [14]:
from aieng.syn_data.synbench.environment.core import Environment, replay_actions
from aieng.syn_data.synbench.environment.hashing import db_hash


oracle_env = replay_actions(domain, cancel_task.evaluation_criteria.actions)
target_hash = db_hash(oracle_env.db)

print("Order status after oracle replay:", oracle_env.db["orders"]["ord_1001"]["status"])
print("Target DB hash:", target_hash[:32], "...")

Order status after oracle replay: canceled
Target DB hash: 39529cffe1e5eadb8f98600d7c373048 ...


In [15]:
# Wrong action: try cancel on a shipped order (should fail at dispatch)
from aieng.syn_data.synbench.schemas.actions import Action


bad_env = Environment(domain)
try:
    bad_env.dispatch(Action(name="cancel_order", arguments={"order_id": "ord_1002"}))
except Exception as e:
    print("Expected error:", e)

Expected error: Cannot cancel order with status: shipped


---
## Step 4 — Generate synthetic tasks

**Generation** asks an LLM for a new `Task`: user scenario + oracle actions.

`ConstraintSampler` picks:
1. `task_type` 
2. a record from the primarily collection (`orders` for the retail domain) + the user ID related to that order (`entities`)
3. a `personality_style` from `user_simulator.yaml` → `personality_styles`

The generator fills `user_name` (from the related user row) and `personality_style` separately, keeps full ids/args in `instructions`, and styles an incomplete `initial_message` so the agent must elicit details.


In [16]:
from aieng.syn_data.synbench.generation.sampler import ConstraintSampler


# Sample task_type, entity IDs, and personality_style together
sampler = ConstraintSampler(domain, seed=42)
constraints = sampler.sample()
print("Sampled constraints:")
pprint.pprint(vars(constraints), sort_dicts=False)
print("\nPersonality style name:", (constraints.personality_style or {}).get("name"))

Sampled constraints:
{'task_type': 'inquiry',
 'entities': {'order_id': 'ord_1001', 'user_id': 'user_alice'},
 'allow_write': False,
 'primary_id': 'ord_1001',
 'entity_context': {'order_id': 'ord_1001',
                    'user_id': 'user_alice',
                    'status': 'pending',
                    'shipping_address': '123 Main St, Boston, MA',
                    'users': {'name': 'Alice Chen',
                              'email': 'alice@example.com'}},
 'personality_style': {'name': 'rule_breaker',
                       'description': 'Tries to bend or break policy; requests '
                                      'exceptions, refunds, or cancellations '
                                      'that may not be allowed, and argues '
                                      'when refused.'}}

Personality style name: rule_breaker


### Generate one task given the sampled constraints

In [17]:
from aieng.syn_data.synbench.generation.generator import TrajectoryGenerator


# Generate one draft
gen = TrajectoryGenerator(domain, seed=42)
draft = gen.generate_one("notebook_draft_001", constraints)
pprint.pprint(draft.model_dump(mode="json"), sort_dicts=False)

{'id': 'notebook_draft_001',
 'description': 'Inquire about the status of a specific order',
 'task_type': 'inquiry',
 'user_scenario': {'user_name': 'Alice Chen',
                   'personality_style': 'rule_breaker',
                   'instructions': 'Inquire about the status of order '
                                   'ord_1001. If the agent asks for '
                                   'identification, provide the name Alice '
                                   'Chen. If the agent asks for the order '
                                   'number, provide ord_1001. If the agent '
                                   'refuses any request, complain that you are '
                                   'a VIP and deserve better service.',
                   'initial_message': 'I need an update on my package, and I '
                                      "don't care about your standard wait "
                                      'times. Just tell me where my stuff is '
                    

In [18]:
from aieng.syn_data.synbench.generation.generator import GenerationRun


# Generate N tasks
run = GenerationRun(domain, seed=42)
# run.run_and_verify(n=3) would generate and verify in one step instead.
drafts = run.run(n=3)
# Print all of the drafts
for i, draft in enumerate(drafts):
    print(f"Draft number {i + 1}:")
    pprint.pprint(draft.model_dump(mode="json"), sort_dicts=False)
    print("\n")

Draft number 1:
{'id': 'gen_a7caeeea_0',
 'description': 'Look up order details for a customer inquiry',
 'task_type': 'inquiry',
 'user_scenario': {'user_name': 'Alice Chen',
                   'personality_style': 'rule_breaker',
                   'instructions': 'Inquire about the status of order '
                                   'ord_1001. If the agent asks for the order '
                                   'ID, provide ord_1001. If they ask for your '
                                   'name, provide Alice Chen.',
                   'initial_message': "I need to know what's happening with my "
                                      "stuff right now. Just tell me what's "
                                      "going on with it, I'm tired of waiting "
                                      'for things to move.'},
 'evaluation_criteria': {'actions': [{'name': 'find_user_id',
                                      'arguments': {'name': 'Alice Chen'}},
                                

---
## Step 5 — Verification pipeline (quality gate) overview

Before a draft becomes a benchmark task, three checks run in `verify_draft()`:

1. Domain-specific **policy rules** if any exist — e.g. in the retail domain, `cancel` oracles must target a pending order. These rules live in `verify.py`.
2. **task_type validation** — `allow_write` in `task_types.yaml`: read-only types must not use WRITE tools; write types need at least one.
3. **Replay** — oracle actions must execute without error; compute `target_db_hash`

In [19]:
# Verify the generated tasks, and save them in OUT_DIR/tasks.json
verified_tasks, rejected_drafts = run.verify_drafts(drafts)
print(f" ({len(verified_tasks)} / 3) passed the verification pipeline.")
tasks_path = run.write_tasks(verified_tasks, OUT_DIR)
print(f"Written to: {tasks_path}")
print("Task IDs:", [t.id for t in verified_tasks])

 (3 / 3) passed the verification pipeline.
Written to: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_retail/tasks.json
Task IDs: ['gen_a7caeeea_0', 'gen_e6b7a060_1', 'gen_11483642_2']


In [20]:
# Inspect the rejected tasks if any
if rejected_drafts:
    print("\nRejected tasks:")
    for i, (draft, reason) in enumerate(rejected_drafts, 1):
        print(f"  {i}. {draft.id} ({reason})")
        print(f"    User scenario: {draft.user_scenario.initial_message}")
        print(f"    Oracle actions: {draft.evaluation_criteria.actions}")
else:
    print("No rejected tasks.")

No rejected tasks.


# Step 6 - Load previously generated tasks, re-run verification
Before running evaluation, we need to load the generated tasks. It might worth running verification again in case we are not sure if it was done before.

In [21]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

In [22]:
# Re-verify all tasks in the saved file (same as `synbench verify`)
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


task_path = OUT_DIR / "tasks.json"
with open(task_path) as f:
    saved = json.load(f)


for item in saved["tasks"]:
    task = Task.model_validate(item)
    # Verify a single task
    verification_result = verify_draft(domain, task)
    # Verification report includes:
    # - passed: bool
    # - errors: list[str]
    # - target_db_hash: str | None
    # - warnings: list[str]
    status = "OK" if verification_result.verification_report.passed else "FAIL"
    print(f"  [{status}] {task.id}")

  [OK] gen_a7caeeea_0
  [OK] gen_e6b7a060_1
  [OK] gen_11483642_2


In [23]:
# Inspect one task verification
from aieng.syn_data.synbench.verification.pipeline import verify_draft


# Load the first task as an example
task = Task.model_validate(saved["tasks"][0])

result = verify_draft(domain, task)
report = result.verification_report

print("Passed:", report.passed)
print("Errors:", report.errors)
print(
    "Target DB hash:",
    report.target_db_hash[:32] if report.target_db_hash else None,
    "...",
)
print("\nVerified task ready for eval:", result.task.id)

Passed: True
Errors: []
Target DB hash: 9970b161e0dd6be706f2067662f47aef ...

Verified task ready for eval: gen_a7caeeea_0
